# 5. Original linguistic features

## Objective

Grammar scoring is partly reflected in how much a speaker says, how varied and repetitive the wording is, how sentences are formed, and how often disfluency or punctuation cues appear in the transcript. This notebook converts cached transcripts into the original handcrafted feature set.

The feature groups are: lexical volume (`word_count`, `unique_word_count`, `char_count`); sentence structure (`sentence_count`, `avg_sentence_length`, `sentence_length_std`); lexical diversity (`type_token_ratio`, `avg_word_length`); repetition and disfluency (`repetition_ratio`, `filler_count`); and punctuation cues (`question_count`, `exclamation_count`). Missing transcripts are filled with empty strings before extraction, yielding zero-valued features for empty text.

In [ ]:
import pandas as pd
import numpy as np

PROJECT_ROOT = r"C:\Users\rudra\Desktop\SHL-Grammar-Scoring"

transcripts = pd.read_csv(
    PROJECT_ROOT + r"\outputs\train_transcripts.csv"
)

transcripts["transcript"] = transcripts["transcript"].fillna("")

print("Shape:", transcripts.shape)
print(transcripts["label"].value_counts().sort_index())

In [ ]:
import re
import string

def extract_basic_features(text):
    text = str(text).strip()

    if not text:
        return {
            "word_count": 0,
            "unique_word_count": 0,
            "sentence_count": 0,
            "avg_sentence_length": 0,
            "sentence_length_std": 0,
            "type_token_ratio": 0,
            "char_count": 0,
            "avg_word_length": 0,
            "repetition_ratio": 0,
            "filler_count": 0,
            "question_count": 0,
            "exclamation_count": 0
        }

    words = re.findall(r"\b[a-zA-Z']+\b", text.lower())

    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    word_count = len(words)
    unique_word_count = len(set(words))

    sentence_lengths = [
        len(re.findall(r"\b[a-zA-Z']+\b", s))
        for s in sentences
    ]

    filler_words = {
        "um", "uh", "erm", "hmm", "like",
        "you", "know", "actually", "basically"
    }

    filler_count = sum(
        1 for word in words
        if word in filler_words
    )

    repeated_count = word_count - unique_word_count

    return {
        "word_count": word_count,
        "unique_word_count": unique_word_count,
        "sentence_count": len(sentences),
        "avg_sentence_length": np.mean(sentence_lengths) if sentence_lengths else 0,
        "sentence_length_std": np.std(sentence_lengths) if sentence_lengths else 0,
        "type_token_ratio": unique_word_count / word_count if word_count else 0,
        "char_count": len(text),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "repetition_ratio": repeated_count / word_count if word_count else 0,
        "filler_count": filler_count,
        "question_count": text.count("?"),
        "exclamation_count": text.count("!")
    }

In [ ]:
feature_rows = transcripts["transcript"].apply(extract_basic_features)

features = pd.DataFrame(feature_rows.tolist())

features["filename"] = transcripts["filename"]
features["label"] = transcripts["label"]

print("Feature shape:", features.shape)

display(features.head())

In [ ]:
print(
    features.corr(numeric_only=True)["label"]
    .sort_values(ascending=False)
)

In [ ]:
output_path = PROJECT_ROOT + r"\outputs\linguistic_features.csv"

features.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
# Inspect feature distributions and their Pearson association with the target.
# This uses the feature table already created above and does not alter extraction.
import matplotlib.pyplot as plt

numeric_features = features.select_dtypes(include=["number"]).drop(columns=["label"], errors="ignore")
correlations = numeric_features.join(features["label"]).corr(numeric_only=True)["label"].drop("label").sort_values()
print(correlations.to_string())

numeric_features.hist(figsize=(14, 10), bins=20)
plt.suptitle("Original linguistic feature distributions")
plt.tight_layout()
plt.show()

correlations.plot.barh(figsize=(8, 5), title="Feature correlation with grammar score")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## Interpretation

The tables and plots show the scale and target association of every original feature without implying causation. These diagnostics help identify constant or redundant columns and motivate the later preprocessing step that removes constant features before modeling.